In [ ]:
!pip install -U transformers accelerate datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
POSITIVE_PATH = "/content/positive_reviews.csv"
NEGATIVE_PATH = "/content/negative_reviews.csv"
# --- 2. Загрузка данных ---
df_pos = pd.read_csv(POSITIVE_PATH)
df_neg = pd.read_csv(NEGATIVE_PATH)

# Ожидаемые колонки: id, category, text
assert {"id", "category", "text"}.issubset(df_pos.columns)
assert {"id", "category", "text"}.issubset(df_neg.columns)

df_pos["label"] = 1  # хорошие отзывы
df_neg["label"] = 0  # плохие отзывы

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

print(df["label"].value_counts())

label
1    1601
0    1519
Name: count, dtype: int64


In [ ]:
# Делим датасет на тестовый и валидационный
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

print("Train size:", len(train_df))
print("Val size:  ", len(val_df))

Train size: 2496
Val size:   624


In [ ]:
import transformers
print(transformers.__version__)

4.57.1


In [ ]:
# Загрузим модель
MODEL_NAME = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ReviewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item


train_dataset = ReviewsDataset(train_df, tokenizer)
val_dataset = ReviewsDataset(val_df, tokenizer)

# Параметры обучения

num_labels = 2
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)

# Метрики

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1_weighted": f1}

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_reviews_cls",
    # без evaluation_strategy/save_strategy и прочего "современного"
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    # Если есть GPU - будет использоваться автоматически
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Обучение

trainer.train()

# Оценка

eval_results = trainer.evaluate()
print("Eval:", eval_results)

# Более подробный отчёт:
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(predictions.label_ids, preds, digits=4))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Step,Training Loss
50,0.357100
100,0.170700
150,0.225000
200,0.122900
250,0.063400
300,0.030300
350,0.042800
400,0.019800
450,0.016700
500,0.028300


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Eval: {'eval_loss': 0.052273061126470566, 'eval_accuracy': 0.9919871794871795, 'eval_f1_weighted': 0.9919868293908033, 'eval_runtime': 510.1442, 'eval_samples_per_second': 1.223, 'eval_steps_per_second': 0.076, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

           0     0.9934    0.9901    0.9918       304
           1     0.9907    0.9938    0.9922       320

    accuracy                         0.9920       624
   macro avg     0.9920    0.9919    0.9920       624
weighted avg     0.9920    0.9920    0.9920       624



In [ ]:
import shap
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_pos(texts):
    """
    Возвращает только вероятность положительного класса P(class=1).
    Это 1D-выход, с которым SHAP отлично дружит.
    """
    # SHAP может передавать строку, список строк или список токенов
    if isinstance(texts, str):
        texts = [texts]

    processed_texts = []
    for t in texts:
        if isinstance(t, list):
            processed_texts.append(" ".join(t))
        else:
            processed_texts.append(str(t))

    enc = tokenizer(
        processed_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        # берём вероятность класса 1 (positive)
        pos_probs = probs[:, 1]
    return pos_probs

In [ ]:
# Разбиваем текст по небуквенным символам
masker = shap.maskers.Text(tokenizer=r"\W+")

explainer = shap.Explainer(
    predict_pos,
    masker=masker,
)

In [ ]:
# Берём несколько отзывов из валидации
sample_texts = val_df["text"].sample(5, random_state=42).tolist()

# Считаем shap-значения (это может занять несколько минут)
shap_values = explainer(sample_texts)

# Показываем объяснение для первого отзыва
shap.plots.text(shap_values[0])

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  20%|██        | 1/5 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  60%|██████    | 3/5 [03:44<02:40, 80.04s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  80%|████████  | 4/5 [05:08<01:21, 81.86s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 5/5 [05:46<00:00, 65.46s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 6it [08:17, 99.40s/it]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
SAVE_DIR = "/content/drive/MyDrive/bert_reviews_model"  # можешь назвать по-другому

# сохранить модель и токенайзер
trainer.save_model(SAVE_DIR)            # сохраняет веса и config
tokenizer.save_pretrained(SAVE_DIR)     # сохраняет токенайзер

print("Сохранено в:", SAVE_DIR)

Сохранено в: /content/drive/MyDrive/bert_reviews_model
